<a href="https://colab.research.google.com/github/Laiba-Tahir29/Machine_learning_internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

print("Setup done.")

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
#staleness
signal1 = con.sql(f"""
    WITH page_activity AS (
        SELECT
            content_hash_id,
            COUNT(DISTINCT report_date) as active_days,
            SUM(gsc_impressions) as total_impressions,
            SUM(gsc_clicks) as total_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        CASE
            WHEN active_days <= 10 THEN 'low_activity (stale-like)'
            WHEN active_days <= 20 THEN 'medium_activity'
            ELSE 'high_activity (fresh-like)'
        END as activity_bucket,
        COUNT(*) as n,
        ROUND(AVG(total_impressions), 1) as avg_impressions,
        ROUND(AVG(total_clicks), 1) as avg_clicks
    FROM page_activity
    GROUP BY activity_bucket
    ORDER BY avg_impressions DESC
""").df()

print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

              activity_bucket       n  avg_impressions  avg_clicks
0  high_activity (fresh-like)  103225           2637.4         7.6
1             medium_activity   25065            283.7         1.2
2   low_activity (stale-like)   48448             26.9         0.1


verdict for 1-confirmed

In [3]:
#ctr vs position
signal2 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 10 AND gsc_avg_position > 0 THEN 'top_10_position'
            WHEN gsc_avg_position <= 20 THEN 'position_11_20'
            ELSE 'position_20_plus'
        END as position_bucket,
        COUNT(*) as n,
        ROUND(AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions END), 4) as avg_ctr
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 10
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    position_bucket        n  avg_ctr
0    position_11_20   365148   0.0026
1  position_20_plus   445678   0.0013
2   top_10_position  1336703   0.0034


verdict for 2-confirmed

RULE:
## My rule and its reason codes

**Signal check summary:**

Both signals I tested were CONFIRMED with real data:
- Signal 1 (Staleness): High-activity pages get ~98x more impressions
  than low-activity (stale) pages (2637.4 vs 26.9 avg impressions).
- Signal 2 (Position vs CTR): CTR drops as position gets worse — top-10
  pages average 0.0034 CTR, dropping to 0.0013 for position 20+.

**My rule (in plain words):**

Based on these confirmed signals, I will flag two types of pages for review:

1. **Stale-but-visible pages** — pages with low recent activity but still
   real impressions. Since Signal 1 confirms low activity is strongly
   linked to poor performance, these pages are being neglected but still
   have an audience worth serving.

2. **Underperforming top-position pages** — pages that rank well (top 10
   or top 20 position) but whose CTR is below what Signal 2 shows is
   normal for that position tier. Since position strongly predicts CTR,
   a page in a good position with low CTR is a real anomaly, not just
   "expected low performance" — it's likely a title/meta description
   problem, not a ranking problem.

I will NOT flag low-position pages just because their CTR is naturally
lower — Signal 2 shows that's expected behavior, not a fixable problem.

**Reason codes this rule can output:**

- `stale_but_visible` — low recent activity days, but meaningful impressions still coming in
- `top_position_low_ctr` — good position (top 20), but CTR below that position tier's average — likely a title/meta issue
- `low_priority` — neither condition met; page doesn't need urgent review

**Action label per reason code:**
- `stale_but_visible` → action: **refresh**
- `top_position_low_ctr` → action: **rewrite_title_meta**
- `low_priority` → action: **monitor**

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

PULL THE DATA FIRST

In [4]:
page_data = con.sql(f"""
    WITH page_stats AS (
        SELECT
            content_hash_id,
            client_hash_id,
            COUNT(DISTINCT report_date) as active_days,
            SUM(gsc_impressions) as total_impressions,
            SUM(gsc_clicks) as total_clicks,
            AVG(gsc_avg_position) as avg_position
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        *,
        CASE WHEN total_impressions > 0 THEN total_clicks * 1.0 / total_impressions ELSE 0 END as ctr
    FROM page_stats
""").df()

print(f"Total pages loaded: {len(page_data)}")
page_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages loaded: 176738


,content_hash_id,client_hash_id,active_days,total_impressions,total_clicks,avg_position,ctr
0,content_ac8663da7484669a,client_62f4a7e64f5e0096,17,34.0,0.0,4.909314,0.000000
1,content_39d7361b4945d504,client_62f4a7e64f5e0096,24,77.0,0.0,4.074107,0.000000
2,content_d49a012dcb924e31,client_62f4a7e64f5e0096,31,329.0,0.0,5.177774,0.000000
3,content_614baf2af4330bd7,client_62f4a7e64f5e0096,31,772.0,1.0,4.685335,0.001295
4,content_225dc9235023be5f,client_62f4a7e64f5e0096,31,488.0,1.0,17.148172,0.002049


NOW RANK THEM

In [5]:
import numpy as np
import pandas as pd

def get_expected_ctr(position):
    if position <= 10 and position > 0:
        return 0.0034
    elif position <= 20:
        return 0.0026
    else:
        return 0.0013

page_data["expected_ctr"] = page_data["avg_position"].apply(get_expected_ctr)

def score_and_flag(row):
    is_stale = row["active_days"] <= 10
    is_visible = row["total_impressions"] >= 100
    is_top_position = 0 < row["avg_position"] <= 20
    is_low_ctr = row["ctr"] < row["expected_ctr"] * 0.7

    if is_stale and is_visible:
        reason_code = "stale_but_visible"
        action = "refresh"
        score = row["total_impressions"] * 0.6
    elif is_top_position and is_low_ctr:
        reason_code = "top_position_low_ctr"
        action = "rewrite_title_meta"
        score = row["total_impressions"] * (row["expected_ctr"] - row["ctr"]) * 100
    else:
        reason_code = "low_priority"
        action = "monitor"
        score = 0

    return pd.Series([score, reason_code, action])

page_data[["score", "reason_code", "action"]] = page_data.apply(score_and_flag, axis=1)

ranked_queue = page_data.sort_values("score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

print(f"Total flagged (score > 0): {(ranked_queue['score'] > 0).sum()}")
ranked_queue.head(10)

Total flagged (score > 0): 93422


,content_hash_id,client_hash_id,active_days,total_impressions,total_clicks,avg_position,ctr,expected_ctr,score,reason_code,action,rank
0,content_44f34c0a90047651,client_23a62021009f63c4,31,212404.0,24.0,7.346909,0.000113,0.0034,69817.36,top_position_low_ctr,rewrite_title_meta,1
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,31,134984.0,1.0,4.545582,0.000007,0.0034,45794.56,top_position_low_ctr,rewrite_title_meta,2
2,content_34a70fea29d15f24,client_62f4a7e64f5e0096,31,143019.0,43.0,3.219473,0.000301,0.0034,44326.46,top_position_low_ctr,rewrite_title_meta,3
3,content_fec55986a1868d62,client_73cda7b4e4f265ea,31,124075.0,1.0,9.385150,0.000008,0.0034,42085.50,top_position_low_ctr,rewrite_title_meta,4
4,content_8d7d99f109e19aa2,client_e547b89c05043229,29,203497.0,289.0,2.563756,0.001420,0.0034,40288.98,top_position_low_ctr,rewrite_title_meta,5
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,31,132593.0,83.0,5.789019,0.000626,0.0034,36781.62,top_position_low_ctr,rewrite_title_meta,6
6,content_f6116743b00afc2d,client_62f4a7e64f5e0096,31,107584.0,15.0,9.536301,0.000139,0.0034,35078.56,top_position_low_ctr,rewrite_title_meta,7
7,content_acbcc847f8996314,client_62f4a7e64f5e0096,31,170808.0,262.0,3.361195,0.001534,0.0034,31874.72,top_position_low_ctr,rewrite_title_meta,8
8,content_b99ea6861864dea5,client_62f4a7e64f5e0096,31,194337.0,361.0,4.450106,0.001858,0.0034,29974.58,top_position_low_ctr,rewrite_title_meta,9
9,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,31,89332.0,4.0,7.786219,0.000045,0.0034,29972.88,top_position_low_ctr,rewrite_title_meta,10


In [6]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_cols = ["rank", "content_hash_id", "client_hash_id", "score",
               "reason_code", "action", "total_impressions", "total_clicks",
               "avg_position", "ctr", "active_days"]

ranked_queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved! File shape:", ranked_queue.shape)
print("\nTop 5 rows:")
ranked_queue[output_cols].head()

Saved! File shape: (176738, 12)

Top 5 rows:


,rank,content_hash_id,client_hash_id,score,reason_code,action,total_impressions,total_clicks,avg_position,ctr,active_days
0,1,content_44f34c0a90047651,client_23a62021009f63c4,69817.36,top_position_low_ctr,rewrite_title_meta,212404.0,24.0,7.346909,0.000113,31
1,2,content_8e1334d6356668e3,client_73cda7b4e4f265ea,45794.56,top_position_low_ctr,rewrite_title_meta,134984.0,1.0,4.545582,0.000007,31
2,3,content_34a70fea29d15f24,client_62f4a7e64f5e0096,44326.46,top_position_low_ctr,rewrite_title_meta,143019.0,43.0,3.219473,0.000301,31
3,4,content_fec55986a1868d62,client_73cda7b4e4f265ea,42085.50,top_position_low_ctr,rewrite_title_meta,124075.0,1.0,9.385150,0.000008,31
4,5,content_8d7d99f109e19aa2,client_e547b89c05043229,40288.98,top_position_low_ctr,rewrite_title_meta,203497.0,289.0,2.563756,0.001420,29


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
#lET me check
top20 = ranked_queue[output_cols].head(20)
print(top20.to_string())

    rank           content_hash_id           client_hash_id     score           reason_code              action  total_impressions  total_clicks  avg_position       ctr  active_days
0      1  content_44f34c0a90047651  client_23a62021009f63c4  69817.36  top_position_low_ctr  rewrite_title_meta           212404.0          24.0      7.346909  0.000113           31
1      2  content_8e1334d6356668e3  client_73cda7b4e4f265ea  45794.56  top_position_low_ctr  rewrite_title_meta           134984.0           1.0      4.545582  0.000007           31
2      3  content_34a70fea29d15f24  client_62f4a7e64f5e0096  44326.46  top_position_low_ctr  rewrite_title_meta           143019.0          43.0      3.219473  0.000301           31
3      4  content_fec55986a1868d62  client_73cda7b4e4f265ea  42085.50  top_position_low_ctr  rewrite_title_meta           124075.0           1.0      9.385150  0.000008           31
4      5  content_8d7d99f109e19aa2  client_e547b89c05043229  40288.98  top_position_low_ct

## 3. Top-20 review

**Rank 1 — content_44f34c0a90047651:** Action: rewrite_title_meta.
Position 7.3 (strong), but CTR 0.011% vs ~0.34% expected — huge gap with
212K impressions (large sample, high confidence). Could be wrong if the
page's content doesn't match search intent — a better title won't help
if the content itself isn't what searchers want.

**Rank 2 — content_8e1334d6356668e3:** Action: rewrite_title_meta.
Position 4.5 (very strong), but only 1 click from 135K impressions —
extreme gap, high confidence. Could be wrong if this is a navigational
query where users don't need to click (e.g., they get their answer from
the snippet alone).

**Rank 3 — content_34a70fea29d15f24:** Action: rewrite_title_meta.
Position 3.2 (excellent), CTR 0.03% is far below expected — high
confidence given 143K impressions. Could be wrong if impressions are
inflated by an ambiguous/broad query that isn't truly relevant to this page.

**Rank 4 — content_fec55986a1868d62:** Action: rewrite_title_meta.
Position 9.4, only 1 click from 124K impressions — very high confidence
red flag. Could be wrong if this page ranks for a very broad/generic term
where most searchers aren't the target audience.

**Rank 5 — content_8d7d99f109e19aa2:** Action: rewrite_title_meta.
Position 2.6 (near-top), CTR 0.14% still below the ~0.34% expected for
top-10 — moderate-high confidence. Could be wrong since this page already
gets 289 clicks; the "problem" may be smaller in practice than the score
suggests.

**Rank 6 — content_7c6373141eae744a:** Action: rewrite_title_meta.
Position 5.8, CTR 0.06% vs 0.34% expected — high confidence. Could be
wrong if seasonal/trending demand inflated impressions temporarily
without real buyer intent.

**Rank 7 — content_f6116743b00afc2d:** Action: rewrite_title_meta.
Position 9.5, CTR 0.014% — very low despite decent position, high
confidence. Could be wrong if position 9.5 sits right at page-one/two
boundary, where visibility (and thus true opportunity) is inconsistent.

**Rank 8 — content_acbcc847f8996314:** Action: rewrite_title_meta.
Position 3.4, CTR 0.15% below the ~0.34% expected — moderate confidence,
gap is real but smaller than top rows. Could be wrong if this topic
naturally has lower click intent (e.g., informational vs. transactional).

**Rank 9 — content_b99ea6861864dea5:** Action: rewrite_title_meta.
Position 4.5, CTR 0.19% — gap exists but this page already gets 361
clicks, moderate confidence. Could be wrong because the absolute click
volume is already decent — a rewrite has less room to add value here.

**Rank 10 — content_cd3d932d4e1c8db0:** Action: rewrite_title_meta.
Position 7.8, only 4 clicks from 89K impressions — high confidence red
flag. Could be wrong if this page's title already matches intent well
but the meta description specifically is the issue — rewriting the wrong
part won't fix it.

**Rank 11 — content_f43118e089ecc69a:** Action: rewrite_title_meta.
Position 5.0, CTR 0.14%, already has 191 clicks — moderate confidence,
smaller gap than top-10 rows. Could be wrong if 191 clicks is already
close to this topic's natural ceiling.

**Rank 12 — content_046fc480045b88f5:** Action: rewrite_title_meta.
Position 7.3, but only 25 active days (not full month) — moderate
confidence, since incomplete tracking could understate true performance.
Could be wrong if the missing days would have shown much better (or
worse) numbers, skewing this score.

**Rank 13 — content_9540d884af3e41fd:** Action: rewrite_title_meta.
Position 7.8, CTR 0.013% from 82K impressions — high confidence, full
month of data. Could be wrong if this is a duplicate/near-duplicate of
another ranking page, splitting clicks unfairly.

**Rank 14 — content_425715547c6a3ea8:** Action: rewrite_title_meta.
Position 6.4, only 3 clicks from 71K impressions — high confidence red
flag. Could be wrong if the page is mid-redesign or recently changed,
making the CTR data temporarily unrepresentative.

**Rank 15 — content_306bc78dff1eb683:** Action: rewrite_title_meta.
Position 1.5 (near #1!), yet CTR only 0.04% — surprising and high
confidence given the excellent position. Could be wrong if this ranks
for a query where the SERP itself answers the question (featured
snippet), reducing clicks regardless of title quality.

**Rank 16 — content_e578ac84778da489:** Action: rewrite_title_meta.
Position 4.1, CTR 0.14%, already 163 clicks — moderate confidence, real
but smaller gap. Could be wrong if seasonal dips in this topic are
temporary and will self-correct.

**Rank 17 — content_36fc1ee501ec072d:** Action: rewrite_title_meta.
Position 6.5, CTR 0.02% from 73K impressions — high confidence. Could be
wrong if this page targets a very specific niche query where low CTR is
simply normal for that audience.

**Rank 18 — content_4977e90c4d93cf9f:** Action: rewrite_title_meta.
Position 7.4, CTR 0.04%, moderate confidence given smaller click count
(28). Could be wrong if 28 clicks is actually a healthy number for this
page's niche topic.

**Rank 19 — content_9c057b66c30a3abb:** Action: rewrite_title_meta.
Position 11.2 — just outside top-10 — CTR 0.001% (only 1 click from 84K
impressions), high confidence red flag. Could be wrong since this page
sits at the position-10/11 boundary, where my expected_ctr bucket (0.0026)
may not perfectly apply.

**Rank 20 — content_9ef3d7516483e665:** Action: rewrite_title_meta.
Position 2.5 (near-top), CTR 0.10% vs ~0.34% expected, 29 active days
(near-full month) — high confidence. Could be wrong if recent algorithm
or SERP layout changes reduced clicks industry-wide, not just for this page.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak picks (from my top-20 review):**

Several of my top-20 picks have lower confidence than others:

- **Rank 5, 9, 11, 16** — these pages already have decent click volume
  (289, 361, 191, 163 clicks respectively). My score still ranked them
  high because of the raw impression volume, but the actual "gap" a
  rewrite could close is smaller than for pages with near-zero clicks
  (like Rank 2, 4, 14, 19). This suggests my scoring formula may
  over-weight impression volume relative to how much room there really
  is for improvement.

- **Rank 12** — only 25 active days out of 31 in the month. Since my
  score is built from a full month's totals, a page with incomplete
  tracking could be over- or under-scored simply due to missing data,
  not real performance.

- **All 20 rows share the same reason code** (`top_position_low_ctr`) —
  none of my top 20 came from the `stale_but_visible` rule, even though
  I confirmed Signal 1 (staleness) was a strong signal too. This is a
  weakness in my scoring formula: the CTR-gap scoring naturally produces
  much larger numbers (tens of thousands) than the staleness scoring
  (based on impressions × 0.6), so staleness-flagged pages never make it
  into the very top ranks, even when they're legitimate flags.

**Leakage check:**

I confirmed that no product decision flags (health_score, priority_score,
action_type) were used anywhere in this rule — they aren't present in
this dataset. I also confirmed no future-window data was used: all
features (active_days, impressions, clicks, position, CTR) come from the
same March 2026 window I'm scoring against, with no information from
April or later months. My label/target for a future model (which I
haven't built yet) will need its own separate future window, kept apart
from these March-only features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.